# MedNote ICD-10 RAG — Step-by-Step Exploration

> A hands-on walkthrough of the Retrieval-Augmented Generation pipeline that turns
> a doctor's plain-language note into precise, billable **ICD-10-CM** codes.

This notebook re-implements the pipeline from
[`docs/icd10_rag_deep_dive.md`](../icd10_rag_deep_dive.md) and
[`docs/implementation_plan.md`](../implementation_plan.md) **inline**, one cell at a
time, so you can *see* what each stage does to the data. Everything runs against the
**real** CMS 2026 XML sitting in `data/corpus/`.

**Two phases, five steps:**

| Phase | Step | What happens | Doc §|
|-------|------|--------------|-------|
| One-time (offline) | **1 · ETL** | Parse 9.7 MB of nested XML → flat, self-contained code documents; enrich with synonyms + demographic tags | Step 1 |
| One-time (offline) | **2 · Embed & Index** | SapBERT dense vectors + BM25 sparse vectors → Qdrant | Step 2 |
| Runtime (per query) | **3 · Hybrid Retrieval** | Gemini normalizes the phrase → formal term; Dense 0.7 + Sparse 0.3, metadata-filtered → top-15 | Step 3 |
| Runtime (per query) | **4 · Re-Rank** | Gemini (`get_fast_llm`) scores note↔candidate → calibrated top-3 | Step 4 |
| Runtime (per query) | **5 · Specificity** | Expand "unspecified" parents into precise children | Step 5 |

> **Reality check:** the docs say *"~72,000 codes"*. The 2026 files we actually have
> yield **46,881** `<diag>` elements — we print the true numbers as we go rather than
> quoting the doc.

> **What changed in this version:** Steps 3 & 4 now use **Gemini** via the project's
> `get_fast_llm()`. Two failures in the earlier bi-encoder-only run are fixed here —
> colloquial queries like *"heart attack"* that retrieved the wrong family, and a
> `ms-marco` cross-encoder whose negative logits collapsed a clear otitis-media case
> into a false zero-hit. See §3.2 and §4.

## 0 · Setup & Configuration

We read every tunable parameter from `config.yml` through the project's own
`mednote.config` loader — no magic numbers in this notebook. First, make the `src/`
package importable and point at the repo root.

In [50]:
import sys
from pathlib import Path

# Locate the repo root (this notebook lives in docs/notebooks/) and add src/ to path.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "config.yml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo root:", REPO_ROOT)

import os
os.chdir(REPO_ROOT)  # so relative paths in config.yml resolve

# Load .env so GOOGLE_API_KEY is available to get_fast_llm() (Gemini) later.
from dotenv import load_dotenv
load_dotenv()
print("GOOGLE_API_KEY loaded:", bool(os.getenv("GOOGLE_API_KEY")))

from mednote.config import get_config
cfg = get_config()

# Pull the knobs we'll use throughout the pipeline.
print("dense_weight        :", cfg.vector_store.dense_weight)
print("sparse_weight       :", cfg.vector_store.sparse_weight)
print("top_k_retrieve      :", cfg.vector_store.top_k_retrieve)
print("top_k_rerank        :", cfg.vector_store.top_k_rerank)
print("confidence_threshold:", cfg.vector_store.confidence_threshold)
print("embeddings model    :", cfg.embeddings.model)
print("reranker model      :", cfg.reranker.model)
print("fast LLM            :", cfg.llm.fast.provider, cfg.llm.fast.model)

Repo root: d:\Projects\MedNote
GOOGLE_API_KEY loaded: True
dense_weight        : 0.7
sparse_weight       : 0.3
top_k_retrieve      : 15
top_k_rerank        : 3
confidence_threshold: 0.7
embeddings model    : cambridgeltl/SapBERT-from-PubMedBERT-fulltext
reranker model      : cross-encoder/ms-marco-MiniLM-L6-v2
fast LLM            : anthropic claude-haiku-4-5-20251001


In [2]:
# The plan lists data/icd10_source/, but the XML actually lives in data/corpus/.
# Resolve robustly so the notebook works regardless.
CORPUS = REPO_ROOT / "data" / "corpus"
TABULAR_XML = CORPUS / "icd10cm_tabular_2026.xml"
INDEX_XML   = CORPUS / "icd10cm_index_2026.xml"

for f in (TABULAR_XML, INDEX_XML):
    print(f"{f.name:32s} {f.stat().st_size/1e6:5.1f} MB  exists={f.exists()}")

icd10cm_tabular_2026.xml           9.7 MB  exists=True
icd10cm_index_2026.xml             9.7 MB  exists=True


## 1 · Peek at the Raw Data

Before parsing, look at what we're dealing with. Two files, each ~9.7 MB:

- **Tabular** = the dictionary. Look up a code → get its official definition, hierarchy,
  and coding rules (`includes`, `excludes1/2`, `useAdditionalCode`).
- **Index** = the reverse lookup. Look up a *term* ("ear infection") → get pointed at a code.

Both matter: doctors speak Index language, but we must retrieve Tabular codes.

In [3]:
import xml.etree.ElementTree as ET

tab_root = ET.parse(TABULAR_XML).getroot()
print("Tabular root:", tab_root.tag)
print("Direct children:", sorted({c.tag for c in tab_root}))
print("Chapters:", len(tab_root.findall("chapter")))

# Walk down to a real leaf code and show every field ICD-10 attaches to it.
chap = tab_root.findall("chapter")[3]           # Endocrine chapter (E-codes)
print("\nChapter", chap.findtext("name"), "-", chap.findtext("desc"))

def find_diag(root, target):
    for d in root.iter("diag"):
        if d.findtext("name") == target:
            return d
    return None

e11 = find_diag(tab_root, "E11")
print("\n<diag> E11 sub-elements:", [c.tag for c in e11])
print("desc      :", e11.findtext("desc"))
print("includes  :", [n.text for n in e11.find("includes").findall("note")][:3] if e11.find("includes") is not None else [])
print("excludes1 :", [n.text for n in e11.find("excludes1").findall("note")][:3] if e11.find("excludes1") is not None else [])
print("children  :", [c.findtext("name") for c in e11.findall("diag")][:5])

Tabular root: ICD10CM.tabular
Direct children: ['chapter', 'introduction', 'version']
Chapters: 22

Chapter 4 - Endocrine, nutritional and metabolic diseases (E00-E89)

<diag> E11 sub-elements: ['name', 'desc', 'includes', 'useAdditionalCode', 'excludes1', 'diag', 'diag', 'diag', 'diag', 'diag', 'diag', 'diag', 'diag', 'diag', 'diag']
desc      : Type 2 diabetes mellitus
includes  : ['diabetes (mellitus) due to insulin secretory defect', 'diabetes NOS', 'insulin resistant diabetes (mellitus)']
excludes1 : ['diabetes mellitus due to underlying condition (E08.-)', 'drug or chemical induced diabetes mellitus (E09.-)', 'gestational diabetes (O24.4-)']
children  : ['E11.0', 'E11.1', 'E11.2', 'E11.3', 'E11.4']


In [4]:
# tab_root.findall("chapter")[0].findtext("desc")
e10 = find_diag(tab_root, "E10")
# type(e10.find("includes"))
# e10.find("includes").findall("note")[:2]

In [5]:
idx_root = ET.parse(INDEX_XML).getroot()
print("Index root:", idx_root.tag, "| letters:", len(idx_root.findall("letter")))

# The Index nests <term level="N"> under <mainTerm> — walking the path builds a phrase.
def show_mainterm(title_substr):
    for letter in idx_root.findall("letter"):
        for mt in letter.findall("mainTerm"):
            if title_substr.lower() in (mt.findtext("title") or "").lower():
                print("mainTerm:", mt.findtext("title"), "| code:", mt.findtext("code"))
                for t1 in mt.findall("term")[:4]:
                    print(f"  L1 {t1.findtext('title'):20s} code={t1.findtext('code')}")
                    for t2 in t1.findall("term")[:2]:
                        print(f"      L2 {t2.findtext('title'):18s} code={t2.findtext('code')}")
                return
show_mainterm("Diabetes")

Index root: ICD10CM.index | letters: 26
mainTerm: Diabetes, diabetic | code: E11.9
  L1 with                 code=None
      L2 amyotrophy         code=E11.44
      L2 arthropathy NEC    code=E11.618
  L1 brittle              code=None
  L1 bronzed              code=E83.110
  L1 complicating pregnancy code=None


## 2 · Step 1 — ETL: Parse & Enrich

**Goal:** turn nested XML into ~47k *flat, self-contained* documents ready to embed.

> **Critical insight (from the plan):** naïve chunking (split every 500 words) would
> *destroy* this data. Each `<diag>` must become **one document**, enriched with its
> parent hierarchy so it stands alone.

### 2a · The document schema

Every code becomes an `ICD10Code`. The `to_embedding_text()` method is what actually
gets vectorised — it fuses the description, hierarchy, and *three* synonym sources.

In [34]:
from dataclasses import dataclass, field, asdict

@dataclass
class ICD10Code:
    """One ICD-10-CM code as a self-contained document for embedding."""
    code: str
    description: str
    hierarchy_path: str
    chapter: str
    chapter_code: str
    includes: list[str] = field(default_factory=list)
    inclusion_terms: list[str] = field(default_factory=list)
    excludes1: list[str] = field(default_factory=list)
    excludes2: list[str] = field(default_factory=list)
    code_first: list[str] = field(default_factory=list)
    use_additional_code: list[str] = field(default_factory=list)
    parent_code: str | None = None
    children_codes: list[str] = field(default_factory=list)
    index_synonyms: list[str] = field(default_factory=list)  # filled in step 2b
    target_sex: list[str] = field(default_factory=list)      # filled in step 2c
    max_age_days: int | None = None

    def to_embedding_text(self) -> str:
        """Fuse code + description + hierarchy + every synonym source into one string."""
        parts = [f"{self.code}: {self.description}",
                 f"Hierarchy: {self.hierarchy_path}"]
        synonyms = self.includes + self.inclusion_terms + self.index_synonyms
        if synonyms:
            parts.append("Also known as: " + ", ".join(synonyms))
        if self.excludes1:
            parts.append("Excludes: " + ", ".join(self.excludes1[:5]))
        return "\n".join(parts)

print("ICD10Code schema ready:", [f for f in ICD10Code.__dataclass_fields__])

ICD10Code schema ready: ['code', 'description', 'hierarchy_path', 'chapter', 'chapter_code', 'includes', 'inclusion_terms', 'excludes1', 'excludes2', 'code_first', 'use_additional_code', 'parent_code', 'children_codes', 'index_synonyms', 'target_sex', 'max_age_days']


### 2a · Recursive tabular parser

The `<diag>` tree is walked depth-first. As we descend we **accumulate the hierarchy
path** (chapter → section → parent descriptions) and record parent/child links — those
links power the Specificity Check in Step 5.

In [7]:
def _notes(diag, tag: str) -> list[str]:
    """Collect <note> text under a child element like <includes> or <excludes1>."""
    el = diag.find(tag)
    if el is None:
        return []
    return [n.text.strip() for n in el.findall("note") if n.text and n.text.strip()]

def _walk(diag, hierarchy_parts, chapter, chapter_code, parent_code, out):
    code = diag.findtext("name", "").strip()
    desc = diag.findtext("desc", "").strip()
    child_elems = diag.findall("diag")
    out.append(ICD10Code(
        code=code,
        description=desc,
        hierarchy_path=" -> ".join(p for p in hierarchy_parts if p),
        chapter=chapter,
        chapter_code=chapter_code,
        includes=_notes(diag, "includes"),
        inclusion_terms=_notes(diag, "inclusionTerm"),
        excludes1=_notes(diag, "excludes1"),
        excludes2=_notes(diag, "excludes2"),
        code_first=_notes(diag, "codeFirst"),
        use_additional_code=_notes(diag, "useAdditionalCode"),
        parent_code=parent_code,
        children_codes=[c.findtext("name", "").strip() for c in child_elems],
    ))
    # Recurse — child descriptions extend the hierarchy path for their descendants.
    for child in child_elems:
        _walk(child, hierarchy_parts + [desc], chapter, chapter_code, code, out)

def parse_tabular(xml_path) -> list[ICD10Code]:
    root = ET.parse(xml_path).getroot()
    codes: list[ICD10Code] = []
    for chapter in root.findall("chapter"):
        c_code = chapter.findtext("name", "").strip()
        c_desc = chapter.findtext("desc", "").strip()
        for section in chapter.findall("section"):       # no nested sections in 2026 data
            s_desc = section.findtext("desc", "").strip()
            for diag in section.findall("diag"):
                _walk(diag, [c_desc, s_desc], c_desc, c_code, None, codes)
    return codes

In [8]:
codes = parse_tabular(TABULAR_XML)
by_code = {c.code: c for c in codes}
print(f"Parsed {len(codes):,} ICD-10 codes")

# Inspect two of the acceptance-test targets.
for probe in ("G44.2", "E11"):
    c = by_code[probe]
    print(f"\\n=== {c.code}  {c.description} ===")
    print("hierarchy:", c.hierarchy_path)
    print("parent   :", c.parent_code, "| children:", c.children_codes)
    print("includes :", c.includes[:2])

Parsed 46,881 ICD-10 codes
\n=== G44.2  Tension-type headache ===
hierarchy: Diseases of the nervous system (G00-G99) -> Episodic and paroxysmal disorders (G40-G47) -> Other headache syndromes
parent   : G44 | children: ['G44.20', 'G44.21', 'G44.22']
includes : []
\n=== E11  Type 2 diabetes mellitus ===
hierarchy: Endocrine, nutritional and metabolic diseases (E00-E89) -> Diabetes mellitus (E08-E13)
parent   : None | children: ['E11.0', 'E11.1', 'E11.2', 'E11.3', 'E11.4', 'E11.5', 'E11.6', 'E11.8', 'E11.9', 'E11.A']
includes : ['diabetes (mellitus) due to insulin secretory defect', 'diabetes NOS']


In [35]:
# codes[0]
by_code["G44.2"].to_embedding_text()

'G44.2: Tension-type headache\\nHierarchy: Diseases of the nervous system (G00-G99) -> Episodic and paroxysmal disorders (G40-G47) -> Other headache syndromes'

### 2b · Enrich with Index synonyms

The Index file is a **free synonym dictionary** built by human coders. We flatten each
`<mainTerm>`/`<term>` path into a phrase and map it to its code, then merge those phrases
into `index_synonyms`. This is what lets *"ear infection"* find *"Otitis media"*.

In [36]:
def parse_index(xml_path, max_syn: int = 10) -> dict[str, list[str]]:
    """Build {code: [natural-language phrases]} by flattening nested <term> paths."""
    root = ET.parse(xml_path).getroot()
    mapping: dict[str, list[str]] = {}

    def recurse(term, trail):
        title = (term.findtext("title") or "").strip()
        phrase = ", ".join(t for t in trail + [title] if t)
        code = term.findtext("code")
        if code:
            code = code.strip()
            bucket = mapping.setdefault(code, [])
            if phrase and phrase not in bucket and len(bucket) < max_syn:
                bucket.append(phrase)
        for sub in term.findall("term"):
            recurse(sub, trail + [title])

    for letter in root.findall("letter"):
        for mt in letter.findall("mainTerm"):
            recurse(mt, [])
    return mapping

index_synonyms = parse_index(INDEX_XML)
print(f"Index mapped phrases onto {len(index_synonyms):,} distinct codes")

# Merge into the parsed codes.
enriched = 0
for code_str, syns in index_synonyms.items():
    if code_str in by_code:
        by_code[code_str].index_synonyms = syns
        enriched += 1
print(f"Enriched {enriched:,} of {len(codes):,} codes with Index synonyms")

# Show a code that gained natural-language synonyms.
c = by_code["I21.9"]
print(f"\n{c.code} synonyms from Index:", c.index_synonyms[:4])

Index mapped phrases onto 20,347 distinct codes
Enriched 16,390 of 46,881 codes with Index synonyms

I21.9 synonyms from Index: ['Infarct, infarction, myocardium, myocardial', 'Infarct, infarction, myocardium, myocardial, type 1']


### 2c · Metadata tagging (demographic hard-filters)

A 45-year-old man should **never** be shown pregnancy codes. We tag codes by prefix so
the retriever can filter them out *before* scoring.

In [37]:
SEX_RESTRICTIONS = {"O": "female", "N40": "male", "N41": "male", "N42": "male"}
AGE_RESTRICTIONS = {"P": 28}   # perinatal codes → newborns only (max_age_days)

def apply_metadata(codes: list[ICD10Code]) -> None:
    for c in codes:
        for prefix, sex in SEX_RESTRICTIONS.items():
            if c.code.startswith(prefix):
                c.target_sex = [sex]
                break
        for prefix, days in AGE_RESTRICTIONS.items():
            if c.code.startswith(prefix):
                c.max_age_days = days
                break

apply_metadata(codes)
print("Female-only codes:", sum(1 for c in codes if c.target_sex == ["female"]))
print("Male-only   codes:", sum(1 for c in codes if c.target_sex == ["male"]))
print("Perinatal   codes:", sum(1 for c in codes if c.max_age_days is not None))
print("\nExample — O80 target_sex:", by_code.get("O80").target_sex if by_code.get("O80") else "n/a")

Female-only codes: 1791
Male-only   codes: 27
Perinatal   codes: 565

Example — O80 target_sex: ['female']


### 2d · Export to JSONL

One JSON object per line — the hand-off artifact between ETL and indexing. This is the
`data/icd10_processed/icd10_codes.jsonl` the plan's `run_etl.py` would produce.

In [38]:
import json

OUT_DIR = REPO_ROOT / "data" / "icd10_processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "icd10_codes.jsonl"

with OUT_PATH.open("w", encoding="utf-8") as f:
    for c in codes:
        f.write(json.dumps(asdict(c), ensure_ascii=False) + "\\n")
print(f"Wrote {len(codes):,} lines -> {OUT_PATH.relative_to(REPO_ROOT)}")

# This is exactly the text SapBERT will see for G44.2 (tension headache):
print("\n----- to_embedding_text() for G44.2 -----")
print(by_code["G44.2"].to_embedding_text())

Wrote 46,881 lines -> data\icd10_processed\icd10_codes.jsonl

----- to_embedding_text() for G44.2 -----
G44.2: Tension-type headache\nHierarchy: Diseases of the nervous system (G00-G99) -> Episodic and paroxysmal disorders (G40-G47) -> Other headache syndromes


## 3 · Step 2 — Embed & Index

Two vectors per code:

- **Dense (SapBERT, 768-d)** — semantic. Knows *"heart attack" ≈ "myocardial infarction"*.
- **Sparse (BM25)** — lexical. Catches exact acronyms like *COPD*, *STEMI*, *NOS*.

> **Scale note:** embedding all 46,881 codes on CPU is slow. For this exploration we
> build a **subset** (all demo-relevant codes + a random sample). Flip `USE_FULL = True`
> to index everything.

In [13]:
import random
random.seed(0)

# Codes our runtime demos need (targets, their parents, and siblings for specificity).
DEMO_CODES = [
    "I21.9", "I21.0", "I21.01", "I21.4",          # heart attack family
    "G44.2", "G44.20", "G44.21", "G44.1", "R51",  # headache family
    "J06.9", "J44.1", "J44.9",                    # URI / COPD
    "M54.5", "M54.50", "M54.9",                   # low back pain
    "H66.9", "H66.90", "H66.91", "H66.92", "H66.93",  # otitis media laterality
    "H65.9", "E11", "E11.9", "I10", "R07.9", "I20.9", "R10.9",
]
DEMO_CODES = [c for c in DEMO_CODES if c in by_code]

USE_FULL = False
SUBSET_N = 2000

if USE_FULL:
    subset = list(codes)
else:
    pool = [c for c in codes if c.code not in set(DEMO_CODES)]
    sample = random.sample(pool, min(SUBSET_N, len(pool)))
    subset = [by_code[c] for c in DEMO_CODES] + sample
print(f"Indexing subset of {len(subset):,} codes (USE_FULL={USE_FULL})")

Indexing subset of 2,027 codes (USE_FULL=False)


### 3a · Dense embeddings with SapBERT

SapBERT was trained on UMLS synonym pairs, so medical paraphrases land close together.
The proof is in the cosine similarities below.

> **First run downloads ~440 MB** (`cambridgeltl/SapBERT-from-PubMedBERT-fulltext`).

In [14]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer(cfg.embeddings.model)   # SapBERT
print("Loaded:", cfg.embeddings.model, "| dim:", embedder.get_sentence_embedding_dimension())

def cos(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

pairs = embedder.encode(
    ["heart attack", "acute myocardial infarction", "broken leg"],
    normalize_embeddings=True,
)
print(f"\n'heart attack' vs 'acute myocardial infarction': {cos(pairs[0], pairs[1]):.3f}  (should be HIGH)")
print(f"'heart attack' vs 'broken leg'                   : {cos(pairs[0], pairs[2]):.3f}  (should be LOW)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\rauth\AppData\Local\Temp\ipykernel_28724\442025186.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Loaded:", cfg.embeddings.model, "| dim:", embedder.get_sentence_embedding_dimension())


Loaded: cambridgeltl/SapBERT-from-PubMedBERT-fulltext | dim: 768



'heart attack' vs 'acute myocardial infarction': 0.685  (should be HIGH)
'heart attack' vs 'broken leg'                   : 0.352  (should be LOW)


In [39]:
# Embed the whole subset. batch_size comes from config.yml.
subset_texts = [c.to_embedding_text() for c in subset]
dense_vectors = embedder.encode(
    subset_texts,
    batch_size=cfg.embeddings.batch_size,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print("Dense matrix:", dense_vectors.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Dense matrix: (2027, 768)


### 3b · Sparse BM25 vectors

We use `fastembed`'s `Qdrant/bm25` — it emits token indices + IDF-weighted values,
exactly the sparse format Qdrant stores.

In [40]:
from fastembed import SparseTextEmbedding

bm25 = SparseTextEmbedding(model_name="Qdrant/bm25")
sparse_vectors = list(bm25.embed(subset_texts))   # one SparseEmbedding per doc
demo = sparse_vectors[0]
print("Sparse vector for first doc — non-zero terms:", len(demo.indices))
print("sample indices:", demo.indices[:6].tolist())
print("sample values :", [round(float(v), 3) for v in demo.values[:6]])

Sparse vector for first doc — non-zero terms: 22
sample indices: [308249845, 613148321, 1447403130, 422879241, 1576701182, 1672302879]
sample values : [1.542, 1.542, 1.926, 2.027, 2.074, 1.542]


In [41]:
dense_vectors[0].shape , sparse_vectors[0]

((768,),
 SparseEmbedding(values=array([1.54216867, 1.54216867, 1.92612859, 2.02706594, 2.0736377 ,
        1.54216867, 1.54216867, 1.81326465, 1.54216867, 1.54216867,
        1.54216867, 1.54216867, 1.54216867, 1.54216867, 1.54216867,
        1.54216867, 1.54216867, 1.54216867, 1.54216867, 1.81326465,
        1.54216867, 1.54216867]), indices=array([ 308249845,  613148321, 1447403130,  422879241, 1576701182,
        1672302879,  106814587, 1157923234,  574692281, 2095749492,
        1098098095,  228396328, 1918859132,  844567596, 1354500276,
         326725534,  352646395,  634810918,  311264891, 1480931663,
        2045916435, 1810453357])))

### 3c · Build the Qdrant collection

Each point carries **both** a named dense vector (`dense`) and a named sparse vector
(`bm25`), plus a metadata payload. We use an **in-memory** client here so the notebook
never fights the on-disk single-process lock; the real pipeline uses
`QdrantClient(path=cfg.vector_store.local_path)`.

In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")   # notebook-local; swap for path=... in production
COLL = cfg.vector_store.collection_name

client.recreate_collection(
    collection_name=COLL,
    vectors_config={"dense": models.VectorParams(
        size=dense_vectors.shape[1], distance=models.Distance.COSINE)},
    sparse_vectors_config={"bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)},
)

points = []
for i, (c, dv, sv) in enumerate(zip(subset, dense_vectors, sparse_vectors)):
    points.append( .PointStruct(
        id=i,
        vector={
            "dense": dv.tolist(),
            "bm25": models.SparseVector(indices=sv.indices.tolist(), values=sv.values.tolist()),
        },
        payload={
            "code": c.code,
            "description": c.description,
            "hierarchy_path": c.hierarchy_path,
            "children_codes": c.children_codes,
            "parent_code": c.parent_code,
            "target_sex": c.target_sex or ["all"],
            "max_age_days": c.max_age_days,
        },
    ))
client.upsert(COLL, points)
# NOTE: local Qdrant warns that payload indexes are no-ops — filtering still works.
print(f"Upserted {client.count(COLL).count:,} points into '{COLL}'")

C:\Users\rauth\AppData\Local\Temp\ipykernel_29528\1507920564.py:6: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Upserted 2,027 points into 'icd10_codes'


In [44]:
points[0]

PointStruct(id=0, vector={'dense': [-0.07259143888950348, 0.013180574402213097, 0.004583017900586128, 0.007229254115372896, 0.02268332801759243, 0.008921702392399311, 0.006157584488391876, 0.033381056040525436, 0.031579047441482544, -0.0020521855913102627, -0.007990177720785141, 0.019369442015886307, 0.0017701326869428158, 0.003201026702299714, 0.0012239314382895827, 0.038344718515872955, 0.020629679784178734, 0.005099293310195208, -0.09509231895208359, 0.0033691434655338526, 0.021884402260184288, -0.003850187873467803, -0.0061693028546869755, -0.0019290298223495483, -0.01888074167072773, 0.05125141516327858, 0.015118359588086605, 0.010541480034589767, 0.048685017973184586, -0.06933404505252838, -0.012566504999995232, -0.02659178338944912, 0.018134815618395805, 0.09012485295534134, 0.00505100330337882, -0.05790385603904724, 0.018135378137230873, -0.06242148578166962, 0.01932397112250328, 0.030124714598059654, 0.031225234270095825, 0.007203294429928064, -0.020731734111905098, 0.02249484

## 4 · Step 3 — Hybrid Retrieval (LangChain + RRF)

For each clinical entity we run **two LangChain retrievers** over the same Qdrant
collection — a **dense** one (SapBERT) and a **sparse** one (BM25) — and fuse their
results with **Reciprocal Rank Fusion (RRF)** via LangChain's `EnsembleRetriever`.

RRF ignores the raw scores entirely and combines on *rank position*:

```
RRF_score(doc) = Σ_retriever  weight / (K + rank_in_that_list)      # K = 60, rank starts at 1
```

This is exactly why RRF is the right tool here. Dense cosine (0–1) and BM25 (unbounded)
live on **incompatible scales**, so the earlier "min-max normalize each list, then
weighted-average" trick was only ever a stand-in for real fusion. RRF sidesteps the
problem — it never looks at magnitudes, only at order — and our `0.7 / 0.3` weights carry
straight over as the per-retriever RRF weights. A `target_sex` filter is still applied
*before* retrieval, inside each retriever, so demographically-invalid codes never enter
the ranked lists in the first place.

In [45]:
# --- LangChain hybrid-retrieval setup: dense + sparse over the EXISTING collection ---
# Nothing is re-embedded or re-indexed here; we bind LangChain onto the `client`/`COLL`
# built in cell 3c above.
from langchain_core.embeddings import Embeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode

# EnsembleRetriever moved packages in LangChain v1 (langchain -> langchain-classic).
try:
    from langchain.retrievers import EnsembleRetriever          # LangChain <= 0.3
except ImportError:
    from langchain_classic.retrievers import EnsembleRetriever  # LangChain >= 1.0


class SapBERTEmbeddings(Embeddings):
    """Adapter so LangChain can reuse the SapBERT model already loaded in 3a (no re-download)."""

    def __init__(self, model):
        self._m = model

    def embed_documents(self, texts):
        return self._m.encode(texts, normalize_embeddings=True,
                              batch_size=cfg.embeddings.batch_size).tolist()

    def embed_query(self, text):
        return self._m.encode([text], normalize_embeddings=True)[0].tolist()


dense_emb  = SapBERTEmbeddings(embedder)                 # reuse the SapBERT model from 3a
sparse_emb = FastEmbedSparse(model_name="Qdrant/bm25")   # same BM25 as 3b, LangChain-wrapped

# Two views over the SAME collection. content_payload_key="code" makes each returned
# Document.page_content the ICD code itself — our join key back into `by_code`. (The stored
# named vectors are what actually get searched; this only controls what text we read back.)
dense_store = QdrantVectorStore(
    client=client, collection_name=COLL, embedding=dense_emb,
    retrieval_mode=RetrievalMode.DENSE, vector_name="dense", content_payload_key="code")

sparse_store = QdrantVectorStore(
    client=client, collection_name=COLL, sparse_embedding=sparse_emb,
    retrieval_mode=RetrievalMode.SPARSE, sparse_vector_name="bm25", content_payload_key="code")

print("LangChain dense + sparse retrievers bound to existing Qdrant collection.")

LangChain dense + sparse retrievers bound to existing Qdrant collection.


In [46]:
def hybrid_search(query: str, patient_sex: str | None = None, k: int | None = None):
    """Fuse dense + sparse retrieval with weighted RRF via LangChain's EnsembleRetriever.

    Returns a list of dicts with the same keys the manual version produced, so every
    downstream cell (rerank, specificity, pipeline) keeps working unchanged.
    """
    k = k or cfg.vector_store.top_k_retrieve

    sex_filter = None
    if patient_sex:
        sex_filter = models.Filter(must=[models.FieldCondition(
            key="target_sex", match=models.MatchAny(any=["all", patient_sex]))])
    search_kwargs = {"k": k, "filter": sex_filter}

    # EnsembleRetriever == weighted RRF:  score(d) = Σ_r  weight_r / (60 + rank_r(d)).
    # It dedupes across retrievers by page_content, which here IS the ICD code — so the
    # same code found by both dense and sparse merges into one fused entry.
    ensemble = EnsembleRetriever(
        retrievers=[dense_store.as_retriever(search_kwargs=search_kwargs),
                    sparse_store.as_retriever(search_kwargs=search_kwargs)],
        weights=[cfg.vector_store.dense_weight, cfg.vector_store.sparse_weight],
    )

    docs = ensemble.invoke(query)          # RRF-fused, ordered best-first
    results = []
    for d in docs[:k]:
        c = by_code[d.page_content]        # page_content == ICD code (content_payload_key)
        results.append({
            "code": c.code, "description": c.description,
            "hierarchy_path": c.hierarchy_path, "children_codes": c.children_codes,
            "parent_code": c.parent_code, "target_sex": c.target_sex or ["all"],
            "max_age_days": c.max_age_days,
        })
    return results

results = hybrid_search("recurrent tension headache")   # requirements.md Q2 acceptance test
print("Query: 'recurrent tension headache'  (RRF-fused via LangChain EnsembleRetriever)\n")
for rank, r in enumerate(results[:5], 1):
    print(f"  #{rank}  {r['code']:8s} {r['description']}")

Query: 'recurrent tension headache'  (RRF-fused via LangChain EnsembleRetriever)

  #1  G44.21   Episodic tension-type headache
  #2  G44.20   Tension-type headache, unspecified
  #3  G44.2    Tension-type headache
  #4  G44.309  Post-traumatic headache, unspecified, not intractable
  #5  G44.321  Chronic post-traumatic headache, intractable


### 3.2 · Gemini entity normalization (`get_fast_llm`)

Raw retrieval works well when the query already uses formal terminology (the tension
headache demo above), but **breaks on colloquial input**. Watch the next cell: *"heart
attack"* retrieves rheumatic heart disease and atherosclerosis — **not** `I21.9`
(acute MI) — because "heart attack" shares no tokens with the tabular wording and
SapBERT's paraphrase similarity alone isn't decisive.

The plan's Step 3.2 fix is to **normalize the query with an LLM first**. We load the
project's `get_fast_llm()` — now pointed at **Gemini** (`gemini-2.5-flash`) via
`config.yml` — and map the layperson phrase to the formal ICD-10-CM diagnosis term
*before* retrieval.

> **Why `thinking_budget=0`:** Gemini 2.5 models spend output tokens on internal
> reasoning by default, which can truncate short structured replies. We disable it for
> these deterministic extraction/ranking calls so the full answer fits in the budget.

In [51]:
import json
import re

from langchain_core.rate_limiters import InMemoryRateLimiter

from mednote.llm.wrapper import get_fast_llm

# Gemini free tiers are metered per-minute AND per-day. Throttle to ~4/min so a run that
# fires ~10 calls never trips a 429; max_retries adds backoff for transient errors.
rate_limiter = InMemoryRateLimiter(
    requests_per_second=1 / 15,   # 1 request every 15s -> 4/min, under the per-minute cap
    check_every_n_seconds=0.5,
    max_bucket_size=1,
)

# thinking_budget=0 keeps Gemini's short JSON/term replies from being truncated by
# internal reasoning tokens; max_tokens gives headroom for the re-rank JSON in 3.4.
fast_llm = get_fast_llm(
    provider="google",
    model="gemini-flash-lite-latest",
    max_tokens=1024,
    # thinking_budget=0,
    rate_limiter=rate_limiter,
    max_retries=3,
)
print("Fast LLM ready:", type(fast_llm).__name__, "|", fast_llm.model)


def normalize_entity(text: str) -> str:
    """Map a colloquial/transcript phrase to the formal ICD-10-CM diagnosis term so
    dense + sparse retrieval matches the tabular wording (e.g. 'heart attack' ->
    'Myocardial infarction'). Falls back to the original text on an empty reply.

    Uses AIMessage.text (not .content): newer Gemini models return content as a list of
    typed blocks, and .text concatenates the text parts for both old and new formats."""
    prompt = (
        "Rewrite the layperson clinical phrase as the formal ICD-10-CM diagnosis term "
        "used in the tabular list. Return ONLY the term - no codes, punctuation, or notes.\n\n"
        f"Phrase: {text}\nFormal term:"
    )
    out = fast_llm.invoke(prompt).text.strip()
    return out or text


# Throttled to ~4/min, so these few calls take a moment.
for q in ["heart attack", "COPD", "kid has ear infection in both ears"]:
    print(f"  {q!r:45s} -> {normalize_entity(q)!r}")

Fast LLM ready: ChatGoogleGenerativeAI | gemini-flash-lite-latest
  'heart attack'                                -> 'ST elevation myocardial infarction'
  'COPD'                                        -> 'Chronic obstructive pulmonary disease'
  'kid has ear infection in both ears'          -> 'Otitis media, bilateral'


In [52]:
# The same acceptance queries, now normalized through Gemini before retrieval.
# Compare the raw-vs-normalized top hit — "heart attack" is the one that was broken.
for q in ["heart attack", "COPD", "lower back pain"]:
    entity = normalize_entity(q)
    raw_top  = hybrid_search(q)[:1]
    norm_top = hybrid_search(entity)[:3]
    print(f"\nQuery: {q!r}  ->  normalized: {entity!r}")
    print(f"  raw top-1        : {raw_top[0]['code']:8s} {raw_top[0]['description']}")
    for rank, r in enumerate(norm_top, 1):
        print(f"  normalized #{rank}    : {r['code']:8s} {r['description']}")


Query: 'heart attack'  ->  normalized: 'ST elevation myocardial infarction'
  raw top-1        : I01.9    Acute rheumatic heart disease, unspecified
  normalized #1    : I21.01   ST elevation (STEMI) myocardial infarction involving left main coronary artery
  normalized #2    : I21.0    ST elevation (STEMI) myocardial infarction of anterior wall
  normalized #3    : I21.4    Non-ST elevation (NSTEMI) myocardial infarction

Query: 'COPD'  ->  normalized: 'Chronic obstructive pulmonary disease'
  raw top-1        : J44.1    Chronic obstructive pulmonary disease with (acute) exacerbation
  normalized #1    : J44.9    Chronic obstructive pulmonary disease, unspecified
  normalized #2    : J44.1    Chronic obstructive pulmonary disease with (acute) exacerbation
  normalized #3    : Q62.39   Other obstructive defects of renal pelvis and ureter

Query: 'lower back pain'  ->  normalized: 'Low back pain'
  raw top-1        : M54.50   Low back pain, unspecified
  normalized #1    : M54.50   Low

## 5 · Step 4 — LLM Re-Ranking (Gemini)

Retrieval (bi-encoder + BM25) is tuned for **recall** — cast a wide net — but is weak
at **precision**: picking the single most clinically correct code from the top-15.

> **Why not the cross-encoder?** This step originally used
> `cross-encoder/ms-marco-MiniLM-L6-v2`. On this clinical data it failed badly — for
> *"ear pain in both ears, discharge"* it scored every candidate with large **negative**
> logits (−4.7 to −6.7) and ranked `H73.009` (myringitis) above the correct bilateral
> otitis media, collapsing end-to-end confidence to ~0.01 and tripping a false
> zero-hit. `ms-marco` was trained on web-search passages, not terse `CODE: description`
> strings, so its scores are neither accurate nor calibrated here.

We replace it with **`get_fast_llm()` (Gemini)** as the re-ranker: it reads the note and
the candidate list together and returns the best codes with a **calibrated 0–1
confidence** — no sigmoid needed. We score the top-15, keep the top-3.

In [23]:
def rerank(query: str, candidates: list[dict], top_n: int | None = None) -> list[dict]:
    """Gemini re-ranker: read the clinical note + candidate codes together and return
    the best `top_n`, each carrying a calibrated 0-1 `rerank_score`. Only codes present
    in `candidates` are kept (hallucinated codes are dropped), and each returned dict is
    the full candidate dict (children_codes etc.) so Step 5 still works."""
    top_n = top_n or cfg.vector_store.top_k_rerank
    if not candidates:
        return []

    by_cand = {c["code"]: c for c in candidates}
    listing = "\n".join(f"{c['code']}: {c['description']}" for c in candidates)
    prompt = (
        "You are an expert ICD-10-CM coder. From the candidate codes below, pick the "
        f"up to {top_n} that best match the clinical note. Reply with ONLY a JSON array, "
        'best first, each item {"code": <str from candidates>, "confidence": <float 0-1>}. '
        "Confidence = how well the code matches the documented findings.\n\n"
        f"Clinical note: {query}\n\nCandidates:\n{listing}\n\nJSON:"
    )
    raw = fast_llm.invoke(prompt).text          # .text -> handles Gemini block-list content
    match = re.search(r"\[.*\]", raw, re.DOTALL)  # tolerate ```json fences / stray prose
    picks = json.loads(match.group(0)) if match else []

    ranked = []
    for p in picks:
        cand = by_cand.get(p.get("code"))
        if cand is not None:
            cand = dict(cand)
            cand["rerank_score"] = float(p.get("confidence", 0.0))
            ranked.append(cand)
    return ranked[:top_n]


query = "Patient reports ear pain in both ears, fever, and discharge"
candidates = hybrid_search(normalize_entity("acute bilateral otitis media"))
top3 = rerank(query, candidates)
print(f"Query: {query!r}\n")
for r in top3:
    print(f"  conf={r['rerank_score']:.2f}  {r['code']:8s} {r['description']}")

Query: 'Patient reports ear pain in both ears, fever, and discharge'

  conf=0.90  H66.93   Otitis media, unspecified, bilateral
  conf=0.70  H66.9    Otitis media, unspecified


### Zero-hit fallback

If the best score is below `confidence_threshold` (0.7), we **refuse to guess** —
better an honest "assign manually" than a confidently wrong code. Because the Gemini
re-ranker already returns a **calibrated 0–1 confidence**, `confidence()` is now just a
pass-through (the old cross-encoder needed a sigmoid to squash raw logits into (0,1);
the LLM does not).

In [24]:
def confidence(score: float) -> float:
    return float(score)   # Gemini re-ranker already returns a calibrated 0-1 confidence

best = top3[0]
conf = confidence(best["rerank_score"])
print(f"Top code {best['code']} confidence = {conf:.2f} (threshold {cfg.vector_store.confidence_threshold})")
if conf < cfg.vector_store.confidence_threshold:
    print(">> Insufficient data to suggest an accurate ICD-10 code. Please assign manually in EHR.")
else:
    print(">> Accept and pass to specificity check.")

Top code H66.93 confidence = 0.90 (threshold 0.7)
>> Accept and pass to specificity check.


## 6 · Step 5 — Specificity Check

Insurers want the *most specific* code. If a surviving code is an "unspecified" parent
with children, we surface those children so the physician can pick the precise one
(e.g. bilateral vs. left vs. right ear).

In [25]:
def check_and_expand(top_codes: list[dict]) -> list[dict]:
    """For any parent code, attach its children from the store as specificity options."""
    id_by_code = {p.payload["code"]: p.id for p in points}  # notebook-local lookup
    out = []
    for c in top_codes:
        children = c.get("children_codes") or []
        c = dict(c)
        if children:
            recs = client.retrieve(COLL, ids=[id_by_code[ch] for ch in children if ch in id_by_code],
                                   with_payload=True)
            c["specificity_options"] = [{"code": r.payload["code"],
                                         "description": r.payload["description"]} for r in recs]
            c["needs_specificity"] = bool(c["specificity_options"])
        else:
            c["needs_specificity"] = False
        out.append(c)
    return out

# H66.9 (unspecified otitis media) is a parent — it should expand into laterality options.
parent = hybrid_search("otitis media unspecified")
parent = [r for r in parent if r["code"] == "H66.9"] or parent[:1]
for c in check_and_expand(parent):
    print(f"{c['code']}  {c['description']}  needs_specificity={c['needs_specificity']}")
    for opt in c.get("specificity_options", []):
        print(f"    -> {opt['code']}  {opt['description']}")

H66.9  Otitis media, unspecified  needs_specificity=True
    -> H66.90  Otitis media, unspecified, unspecified ear
    -> H66.91  Otitis media, unspecified, right ear
    -> H66.92  Otitis media, unspecified, left ear
    -> H66.93  Otitis media, unspecified, bilateral


## 7 · End-to-End: one function, five steps

Chain everything for the canonical example — *"Kid has ear infection in both ears"* —
mirroring the "How It All Connects" walkthrough in the deep-dive doc.

In [26]:
def rag_pipeline(entity: str, transcript: str, patient_sex: str | None = None):
    norm       = normalize_entity(entity)                             # Step 3.2 (Gemini)
    candidates = hybrid_search(norm, patient_sex=patient_sex)         # Step 3
    top_codes  = rerank(transcript, candidates)                      # Step 4 (Gemini)
    best_conf  = confidence(top_codes[0]["rerank_score"]) if top_codes else 0.0
    if best_conf < cfg.vector_store.confidence_threshold:
        return {"status": "zero_hit",
                "message": "Insufficient data to suggest an accurate ICD-10 code."}
    expanded = check_and_expand(top_codes)                            # Step 5
    return {"status": "ok", "confidence": round(best_conf, 2), "codes": expanded}

out = rag_pipeline(
    entity="acute bilateral otitis media",
    transcript="Kid has ear infection in both ears, fever and some discharge",
    patient_sex="male",
)
print("status:", out["status"], "| confidence:", out.get("confidence"))
for c in out.get("codes", []):
    tag = "  (consider children for specificity)" if c["needs_specificity"] else ""
    print(f"  {c['code']:8s} {c['description']} (Pending Physician Confirmation){tag}")

status: ok | confidence: 0.95
  H66.93   Otitis media, unspecified, bilateral (Pending Physician Confirmation)
  H66.9    Otitis media, unspecified (Pending Physician Confirmation)  (consider children for specificity)


## Where this maps in the real codebase

This notebook implemented, inline, what the plan splits across modules:

| Notebook section | Target module (`src/mednote/rag/…`) |
|------------------|-------------------------------------|
| 2a–2b parsing | `etl/parser.py`, `etl/index_parser.py` |
| 2c metadata | `etl/metadata.py` |
| 2d export | `etl/export.py` |
| 3a dense | `embeddings.py` |
| 3b–3c sparse + Qdrant | `indexer.py` |
| 3.2 entity normalization | `entity_extractor.py` (Gemini `get_fast_llm`) |
| 4 hybrid search | `retriever.py` |
| 5 LLM re-rank | `reranker.py` (Gemini `get_fast_llm`) |
| 6 specificity | `specificity.py` |
| 7 orchestration | `pipeline.py` |

**Next steps to productionize:** run the full ETL (`USE_FULL = True`) into
`data/qdrant_data/` via `QdrantClient(path=…)`, cache/batch the Gemini normalize +
re-rank calls (one round-trip each per query today), and lift each cell into its module
with tests. The `config.yml` `reranker.model` (`ms-marco…`) is now unused by the
runtime path — keep it only if you want the cross-encoder available as a fallback.